# Pipelines — Cross-Validation Mechanism (Phase 3 + Phase 5)

This notebook runs the project's four regression pipelines end to end and shows the full mechanism: k-Fold (k=5) cross-validation repeated over 5 seeds, scaling fitted only inside each training fold (no leakage), and the multi-criteria Top-3 model selection that bridges Pipeline 1 into Pipelines 3 and 4.

Pipelines:
1. **Pipeline 1** — Single-output, Original datasets, all 11 models
2. **Top-3 selection** — multi-criteria composite score (accuracy, generalization gap, fold stability, uncertainty, interpretability)
3. **Pipeline 2** — Multi-output, Original datasets, all 11 models
4. **Pipeline 3** — Single-output, Discrete-Input datasets, Top-3 models only
5. **Pipeline 4** — Multi-output, Discrete-Input datasets, Top-3 models only

All aggregated tables and raw per-fold tables are saved to `results/pipeline{1..4}/`, and the last cell shows the complete combined results.

Setup: make the repo root and `scripts/` importable, silence a harmless sklearn naming warning, and configure pandas to display full tables.

In [ ]:
import sys
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for p in (REPO_ROOT, SCRIPTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

warnings.filterwarnings('ignore', message='X does not have valid feature names.*')
warnings.filterwarnings('ignore', message='X has feature names, but.*')

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from src.config import get_path
from _pipeline_common import run_pipeline, read_top3_models

print('Models registry check:')
from src.models.registry import available_models
print(available_models())

## Pipeline 1 — Single-output, Original datasets, all models

For every dataset, every Level-A output target, every model and every seed, this runs 5-fold CV and aggregates mean/std of R2, RMSE, NRMSE, MAE plus the generalization gap and fold stability. Results are logged to MLflow and saved to `results/pipeline1/`.

In [ ]:
agg1 = run_pipeline(mode='single', discrete=False, pipeline_id=1)
agg1.sort_values(['dataset', 'target', 'test_r2_mean'], ascending=[True, True, False])

## Top-3 multi-criteria model selection (Phase 5)

Ranks models on a weighted composite of accuracy, generalization gap, fold stability, uncertainty and interpretability (weights from `configs/config.yaml`). Accuracy alone never decides the winners.

In [ ]:
top3 = read_top3_models()
ranked = pd.read_csv(get_path('results_dir') / 'top3_models.csv')
print('Top-3 models selected for Pipelines 3 & 4:', top3)
ranked

## Pipeline 2 — Multi-output, Original datasets, all models

All Level-A outputs of a dataset are predicted jointly (models without native multi-output support are wrapped in `MultiOutputRegressor`). Results saved to `results/pipeline2/`.

In [ ]:
agg2 = run_pipeline(mode='multi', discrete=False, pipeline_id=2)
agg2.sort_values(['dataset', 'test_r2_mean'], ascending=[True, False])

## Pipeline 3 — Single-output, Discrete-Input datasets, Top-3 models only

Repeats the single-output protocol on the discretized inputs, restricted to the Top-3 models to keep runtime reasonable, as required by the assignment. Results saved to `results/pipeline3/`.

In [ ]:
agg3 = run_pipeline(mode='single', discrete=True, pipeline_id=3, models=top3)
agg3.sort_values(['dataset', 'target', 'test_r2_mean'], ascending=[True, True, False])

## Pipeline 4 — Multi-output, Discrete-Input datasets, Top-3 models only

Multi-output protocol on discretized inputs, Top-3 models only. Results saved to `results/pipeline4/`.

In [ ]:
agg4 = run_pipeline(mode='multi', discrete=True, pipeline_id=4, models=top3)
agg4.sort_values(['dataset', 'test_r2_mean'], ascending=[True, False])

## Final Summary — Complete Cross-Validation Results (All 4 Pipelines)

All four aggregated tables combined, displayed in full, and saved as one `pipelines_summary.csv` under `results/`.

In [ ]:
agg1_tagged = agg1.copy(); agg1_tagged.insert(0, 'pipeline', 'pipeline1_single_original')
agg2_tagged = agg2.copy(); agg2_tagged.insert(0, 'pipeline', 'pipeline2_multi_original')
agg3_tagged = agg3.copy(); agg3_tagged.insert(0, 'pipeline', 'pipeline3_single_discrete')
agg4_tagged = agg4.copy(); agg4_tagged.insert(0, 'pipeline', 'pipeline4_multi_discrete')

summary = pd.concat([agg1_tagged, agg2_tagged, agg3_tagged, agg4_tagged], ignore_index=True)
summary_path = get_path('results_dir') / 'pipelines_summary.csv'
summary.to_csv(summary_path, index=False)

print('=' * 90)
print('COMBINED AGGREGATED RESULTS — Pipelines 1-4')
print('=' * 90)
display(summary)

print()
print('=' * 90)
print('TOP-3 MODELS (selected from Pipeline 1, used in Pipelines 3 & 4):', top3)
print('=' * 90)

print()
print('All pipeline results (fold-level + aggregated CSVs) saved to:', get_path('results_dir').resolve())
print('Combined summary saved to:', summary_path.resolve())